task1


In [24]:
import pandas as pd

df = pd.read_csv("../data/lab4_dirty_tweets.csv")

print(df.head())
print("==================")
print(df.shape)
print("==================")
print(df.info())

print("==================")
print(df.describe(include="all"))

  tweet_id        created_at username platform  \
0     t001    2026/8/1 08:00    Alice      Web   
1     t002          2026/8/2   bob_22      web   
2     t003        Aug 3 2026   @Carol      WEB   
3     t004  08/04/2026 12:10   david    Mobile   
4     t005    2026/8/5 12:28     Emma   mobile   

                                          tweet_text likes  retweets  \
0  Loved the new dashboard! It is fast, clear, an...     0         0   
1  The update is terrible... app crashed twice. h...    37        11   
2  Trying the new feature today. Not sure what I ...    74        22   
3  Great work @team!!! Version 2.0 feels much smo...   111        33   
4  Why is login still broken??? This is so frustr...   148        44   

         country  
0             US  
1             us  
2             uk  
3             CA  
4  United States  
(50, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
--

task2

b

In [25]:
print(df.isna().sum())
print(df[df.isna().any(axis=1)])

df = df.dropna(subset=["tweet_text"])
df["retweets"] = df["retweets"].fillna(0)

print(df.isna().sum())
print(df[df.isna().any(axis=1)])

tweet_id      0
created_at    0
username      0
platform      0
tweet_text    1
likes         0
retweets      0
country       0
dtype: int64
   tweet_id created_at username platform tweet_text likes  retweets country
25     t026   2026/8/2    frank   MOBILE        NaN    85        65      UK
tweet_id      0
created_at    0
username      0
platform      0
tweet_text    0
likes         0
retweets      0
country       0
dtype: int64
Empty DataFrame
Columns: [tweet_id, created_at, username, platform, tweet_text, likes, retweets, country]
Index: []


Task3

In [26]:
print(df.duplicated().sum())
print(df[df.duplicated(keep=False)])

df = df.drop_duplicates()



2
   tweet_id        created_at username platform  \
3      t004  08/04/2026 12:10   david    Mobile   
7      t008  08/08/2026 12:10    henry      IOS   
48     t004  08/04/2026 12:10   david    Mobile   
49     t008  08/08/2026 12:10    henry      IOS   

                                           tweet_text likes  retweets country  
3   Great work @team!!! Version 2.0 feels much smo...   111        33      CA  
7    Support never replied to my email. disappointed.   259         7     USA  
48  Great work @team!!! Version 2.0 feels much smo...   111        33      CA  
49   Support never replied to my email. disappointed.   259         7     USA  


In [27]:
print(
    df[df.duplicated(
        subset=["tweet_id"],
        keep=False
    )]
)

df = df.drop_duplicates(
    subset=["tweet_id"],
    keep="first"
)

Empty DataFrame
Columns: [tweet_id, created_at, username, platform, tweet_text, likes, retweets, country]
Index: []


Task 4 — Incorrect Data Typesm

In [28]:
df["likes"] = (
    df["likes"].astype(str)
    .str.replace(",", "", regex=False)
)

df["likes"] = pd.to_numeric(
    df["likes"], errors="coerce"
)

df["retweets"] = pd.to_numeric(
    df["retweets"], errors="coerce"
)

df.loc[df["retweets"] < 0, "retweets"] = pd.NA

In [29]:
df["likes"] = df["likes"].fillna(
    df["likes"].median()
)

df["retweets"] = df["retweets"].fillna(0)

Task 5 — Parse Dates

In [30]:
print(df.head())
df["created_at"] = pd.to_datetime(
    df["created_at"],
    errors="coerce",
    format="mixed"
)
print(df.head())

df = df.dropna(subset=["created_at"])

df["date"] = df["created_at"].dt.date
df["hour"] = df["created_at"].dt.hour
df["weekday"] = df["created_at"].dt.day_name()

print(df.head())

  tweet_id        created_at username platform  \
0     t001    2026/8/1 08:00    Alice      Web   
1     t002          2026/8/2   bob_22      web   
2     t003        Aug 3 2026   @Carol      WEB   
3     t004  08/04/2026 12:10   david    Mobile   
4     t005    2026/8/5 12:28     Emma   mobile   

                                          tweet_text  likes  retweets  \
0  Loved the new dashboard! It is fast, clear, an...    0.0       0.0   
1  The update is terrible... app crashed twice. h...   37.0      11.0   
2  Trying the new feature today. Not sure what I ...   74.0      22.0   
3  Great work @team!!! Version 2.0 feels much smo...  111.0      33.0   
4  Why is login still broken??? This is so frustr...  148.0      44.0   

         country  
0             US  
1             us  
2             uk  
3             CA  
4  United States  
  tweet_id          created_at username platform  \
0     t001 2026-08-01 08:00:00    Alice      Web   
1     t002 2026-08-02 00:00:00   bob_22   



Task 6 — Standardize Categories and Strings



In [31]:
df["platform"] = (
    df["platform"].astype("string")
    .str.strip()
    .str.lower()
)

platform_map = {
    "web": "Web",
    "mobile": "Mobile",
    "ios": "iOS",
    "android": "Android"
}

df["platform"] = df["platform"].map(platform_map)

country_map = {
    "US": "United States",
    "USA": "United States",
    "United States": "United States",
    "us": "United States",
    "U.S.": "United States",
    "UK": "United Kingdom",
    "uk": "United Kingdom",
    "United Kingdom": "United Kingdom",
    "Canada": "Canada",
    "CA": "Canada"
}

df["country"] = df["country"].map(country_map)

df["username"] = (
    df["username"].astype("string")
    .str.strip()
    .str.replace(r"^@", "", regex=True)
    .str.lower()
)

df["tweet_text"] = (
    df["tweet_text"].astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["tweet_text_raw"] = df["tweet_text"]

# df["sentiment_raw"] = (
#     df["sentiment_raw"].astype("string")
#     .str.strip()
#     .str.lower()
# )

# sentiment_map = {
#     "positive": "Positive",
#     "pos": "Positive",
#     "negative": "Negative",
#     "neg": "Negative",
#     "neutral": "Neutral"
# }

# df["sentiment_clean"] = (
#     df["sentiment_raw"].map(sentiment_map)
# )

In [32]:
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# =========================================================
# Task 7: Tweet Text Preprocessing
# 流程：
# 原始 tweet
# → Normalization
# → Tokenization
# → Stop-word Removal
# → Lemmatization
# → 得到最终 text_clean
# =========================================================


# ---------------------------------------------------------
# 0. Download required NLTK resources
# 这些资源通常只需要第一次运行时下载
# ---------------------------------------------------------

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# =========================================================
# 7.1 Normalization
# =========================================================

def normalize_tweet(text):

    # 1. 全部转成小写
    # 例如：
    # "Python" / "PYTHON" / "python"
    # → "python"
    text = text.lower()

    # 2. 把 URL 统一替换成 "URL"
    #
    # https?://\S+
    #   https?   → 匹配 http 或 https
    #   ://      → 匹配 ://
    #   \S+      → 匹配后面连续的非空白字符
    #
    # |
    #   → OR，或者
    #
    # www\.\S+
    #   www\.    → 匹配 www.
    #   \S+      → 匹配后续 URL 内容
    #
    # 例如：
    # https://google.com
    # www.github.com
    # → URL
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " URL ",
        text
    )

    # 3. 把用户名 mention 统一替换成 "USER"
    #
    # @      → 匹配 @
    # \w+    → 匹配一个或多个字母、数字、下划线
    #
    # 例如：
    # @alice
    # @team123
    # → USER
    text = re.sub(
        r"@\w+",
        " USER ",
        text
    )

    # 4. 把整数和小数统一替换成 "NUMBER"
    #
    # \b             → 单词边界
    # \d+            → 一个或多个数字
    # (?:\.\d+)?     → 可选的小数部分
    # \b             → 单词边界
    #
    # 例如：
    # 12
    # 2026
    # 3.14
    # → NUMBER
    text = re.sub(
        r"\b\d+(?:\.\d+)?\b",
        " NUMBER ",
        text
    )

    # 5. 把多个连续空白字符统一成一个空格
    #
    # \s  → whitespace，例如空格、tab、换行
    # +   → 一个或多个
    #
    # 例如：
    # "hello      world"
    # → "hello world"
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # 6. 去掉字符串开头和结尾的空白
    return text.strip()


# 对 tweet_text 中每一条 tweet 执行 normalize_tweet()
df["text_normalized"] = (
    df["tweet_text"]
    .apply(normalize_tweet)
)


# =========================================================
# 7.2 Tokenization
# =========================================================

# Tokenization:
# 把一句完整字符串拆成一个个 token
#
# 例如：
# "i love this!"
#
# →
#
# ["i", "love", "this", "!"]

df["tokens"] = (
    df["text_normalized"]
    .apply(word_tokenize)
)


# =========================================================
# 7.3 Stop-word Removal
# =========================================================

# 加载英文 stop words
#
# 例如：
# the, a, an, is, to, of, this ...
#
# 这些词非常常见，
# 对 TF-IDF / 词频分析通常区分度较低

stop_words = set(
    stopwords.words("english")
)


def remove_stopwords(tokens):

    # 遍历每个 token
    # 如果 token 不属于 stop words，就保留下来
    return [
        token
        for token in tokens
        if token not in stop_words
    ]


df["tokens_no_stop"] = (
    df["tokens"]
    .apply(remove_stopwords)
)


# =========================================================
# 7.4 Lemmatization
# =========================================================

# 创建 WordNet Lemmatizer
#
# 用来把词的不同形式尽量还原到基础形式
#
# 例如：
# cars → car
# dogs → dog

lemmatizer = WordNetLemmatizer()


def lemmatize_tokens(tokens):

    return [

        # 对每个 token 做 lemmatization
        lemmatizer.lemmatize(token)

        for token in tokens

        # 只保留纯字母 token
        #
        # 例如：
        # "hello" → True
        # "!"     → False
        # "123"   → False
        if token.isalpha()
    ]


df["tokens_clean"] = (
    df["tokens_no_stop"]
    .apply(lemmatize_tokens)
)


# =========================================================
# 7.5 Join tokens back into text
# =========================================================

# tokens_clean 当前还是 Python list：
#
# ["love", "new", "dashboard"]
#
# 使用 " ".join() 重新拼成字符串：
#
# "love new dashboard"
#
# 后面 CountVectorizer 和 TfidfVectorizer
# 会直接使用这个 text_clean 列

df["text_clean"] = (
    df["tokens_clean"]
    .apply(" ".join)
)


# =========================================================
# 7.6 Compare original and cleaned text
# =========================================================

# 对比原始文本和最终清洗后的文本
print(
    df[
        [
            "tweet_text_raw",
            "text_normalized",
            "tokens",
            "tokens_no_stop",
            "tokens_clean",
            "text_clean"
        ]
    ].head()
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jietong_Zhou\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Jietong_Zhou\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jietong_Zhou\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Jietong_Zhou\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Jietong_Zhou\AppData\Roaming\nltk_data...


                                      tweet_text_raw  \
0  Loved the new dashboard! It is fast, clear, an...   
1  The update is terrible... app crashed twice. h...   
2  Trying the new feature today. Not sure what I ...   
3  Great work @team!!! Version 2.0 feels much smo...   
4  Why is login still broken??? This is so frustr...   

                                     text_normalized  \
0  loved the new dashboard! it is fast, clear, an...   
1   the update is terrible... app crashed twice. URL   
2  trying the new feature today. not sure what i ...   
3  great work USER !!! version NUMBER feels much ...   
4  why is login still broken??? this is so frustr...   

                                              tokens  \
0  [loved, the, new, dashboard, !, it, is, fast, ...   
1  [the, update, is, terrible, ..., app, crashed,...   
2  [trying, the, new, feature, today, ., not, sur...   
3  [great, work, USER, !, !, !, version, NUMBER, ...   
4  [why, is, login, still, broken, ?, ?, ?, th

In [33]:
# =========================================================
# Task 8–10: Vocabulary Pruning + DTM + TF-IDF
#
# 前提：
# Task 7 已经生成了：
# df["text_clean"]
#
# 整体流程：
# text_clean
#     ↓
# Vocabulary Pruning
#     ↓
# DTM (Document-Term Matrix)
#     ↓
# TF-IDF
# =========================================================


import pandas as pd

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)


# =========================================================
# Task 8 — Prune the Vocabulary
# =========================================================

# CountVectorizer 会：
# 1. 从所有 tweet 中建立 vocabulary
# 2. 根据 vocabulary 统计每条 tweet 中每个词出现的次数
#
# 同时我们设置 min_df / max_df 来筛选 vocabulary。

vectorizer = CountVectorizer(

    # 一个词至少出现在 2 条不同的 tweets 中才保留。
    #
    # 注意：
    # 不是“这个词总共出现 2 次”，
    # 而是“至少存在于 2 个 documents 中”。
    #
    # 目的：
    # 删除特别稀有、可能只是噪声的词。
    min_df=2,

    # 如果一个词出现在超过 90% 的 tweets 中，
    # 就把它删除。
    #
    # 因为这种词太常见，
    # 对区分不同 tweets 的帮助较小。
    max_df=0.90,

    # 统一转成小写。
    # Task 7 已经 lowercase 过，
    # 这里再次设置作为保险。
    lowercase=True
)


# fit_transform() 同时完成两件事：
#
# fit:
#   查看所有 df["text_clean"]，
#   学习最终 vocabulary。
#
# transform:
#   按照 vocabulary，
#   把每条 tweet 转换成词频向量。
#
# 得到的 dtm 是一个 sparse matrix（稀疏矩阵）。

dtm = vectorizer.fit_transform(
    df["text_clean"]
)


# 获取最终保留下来的 vocabulary
terms = vectorizer.get_feature_names_out()


# 查看 vocabulary
print("Vocabulary:")
print(terms)

# 查看 vocabulary 大小
print(
    "Vocabulary size:",
    len(terms)
)


# =========================================================
# Task 9 — Create Document-Term Matrix (DTM)
# =========================================================

# DTM = Document-Term Matrix
#
# 在这个 Lab 中：
#
# Row    = 一条 tweet
# Column = 一个 vocabulary term
# Value  = 这个词在该 tweet 中出现多少次
#
# 例如：
#
#             dashboard   update   crash
# Tweet 1         1          0        0
# Tweet 2         0          1        1
# Tweet 3         0          2        0


# 查看 DTM 的形状：
#
# (number of tweets, vocabulary size)

print(
    "DTM shape:",
    dtm.shape
)


# dtm 默认是 sparse matrix，
# 为了方便在这个小 Lab 中查看，
# 可以临时转换成普通 DataFrame。
#
# 注意：
# 如果真实数据非常大，
# 不建议把整个 sparse matrix 转成 dense DataFrame，
# 因为可能占用大量内存。

dtm_df = pd.DataFrame(
    dtm.toarray(),
    columns=vectorizer.get_feature_names_out()
)


# 查看前几行 DTM
print("\nDocument-Term Matrix:")
print(
    dtm_df.head()
)


# =========================================================
# Task 10 — TF-IDF
# =========================================================

# TF-IDF 用来衡量：
#
# “一个词对于某一条 tweet 来说有多重要”
#
# 它同时考虑：
#
# TF:
#   这个词在当前 tweet 中是否比较突出
#
# IDF:
#   这个词在整个 tweet 数据集中是否比较少见
#
# 因此：
#
# 局部突出 + 整体少见
# → TF-IDF 较高


# 创建 TF-IDF Vectorizer
#
# 使用和 Task 8 相同的 vocabulary pruning 规则：
#
# min_df=2
# max_df=0.90

tfidf_vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.90
)


# 学习 vocabulary
# 并把每条 tweet 转换成 TF-IDF vector

tfidf = tfidf_vectorizer.fit_transform(
    df["text_clean"]
)


# 获取 TF-IDF 使用的 vocabulary
tfidf_terms = (
    tfidf_vectorizer
    .get_feature_names_out()
)


# 查看 TF-IDF matrix 的大小
print(
    "\nTF-IDF shape:",
    tfidf.shape
)


# 查看 vocabulary
print(
    "\nTF-IDF vocabulary:"
)

print(
    tfidf_terms
)


# ---------------------------------------------------------
# Convert TF-IDF matrix to DataFrame
# ---------------------------------------------------------

# 每一个 cell 不再是“出现次数”，
# 而是这个词对应的 TF-IDF 权重。

tfidf_df = pd.DataFrame(
    tfidf.toarray(),
    columns=tfidf_terms
)


# 查看前几条 tweet 的 TF-IDF
print(
    "\nTF-IDF Matrix:"
)

print(
    tfidf_df.head()
)

Vocabulary:
['amazing' 'annoying' 'app' 'awful' 'battery' 'beautiful' 'better'
 'broken' 'change' 'chart' 'checking' 'cleaner' 'crashed' 'customer'
 'design' 'disappointed' 'downloaded' 'drain' 'easy' 'email' 'enjoying'
 'error' 'exploring' 'fantastic' 'feature' 'feel' 'file' 'forever'
 'frequent' 'frustrating' 'good' 'great' 'happy' 'installed' 'interface'
 'later' 'latest' 'launched' 'layout' 'loading' 'login' 'love' 'much'
 'need' 'never' 'new' 'note' 'notification' 'number' 'okay' 'open'
 'product' 'quicker' 'really' 'recommendation' 'release' 'replied'
 'search' 'service' 'setting' 'simple' 'smoother' 'sooo' 'still' 'support'
 'sure' 'surprisingly' 'take' 'terrible' 'test' 'thank' 'thanks' 'think'
 'time' 'today' 'trying' 'twice' 'understand' 'update' 'upload' 'url'
 'use' 'user' 'version' 'way' 'work' 'wow' 'yet']
Vocabulary size: 88
DTM shape: (46, 88)

Document-Term Matrix:
   amazing  annoying  app  awful  battery  beautiful  better  broken  change  \
0        0         0    0

# Part B


In [34]:
# =========================================================
# Part B — Sentiment Analysis with RoBERTa
#
# 目标：
# 使用一个已经针对 Twitter / social media
# 做过 sentiment classification fine-tuning 的 RoBERTa 模型，
# 为每条 tweet 预测：
#
# 1. negative probability
# 2. neutral probability
# 3. positive probability
# 4. final sentiment label
# 5. sentiment_score
#
# 注意：
# Part B 不使用 Task 7 的 text_clean，
# 而是从 tweet_text_raw 开始，
# 因为 sentiment analysis 需要尽量保留：
# punctuation / emoji / capitalization / negation 等信息。
# =========================================================


import re
from transformers import pipeline


# =========================================================
# Task 11 — Load RoBERTa Sentiment Model
# =========================================================

# pipeline() 是 Hugging Face 提供的高级接口，
# 可以直接完成：
#
# text
# → tokenizer
# → RoBERTa
# → sentiment probabilities
#
# 不需要我们自己手动调用 tokenizer 和 model。

sentiment_model = pipeline(
    "sentiment-analysis",

    # 使用针对 Twitter sentiment 训练过的 RoBERTa 模型
    model=(
        "cardiffnlp/"
        "twitter-roberta-base-sentiment-latest"
    ),

    # 返回所有 sentiment classes 的 score，
    # 而不是只返回最高的那一个。
    #
    # 也就是：
    # negative
    # neutral
    # positive
    top_k=None
)


# =========================================================
# Optional: Test the model with one tweet
# =========================================================

test_tweet = "I absolutely love this new update!"

test_result = sentiment_model(
    test_tweet
)

print("Test result:")
print(test_result)


# =========================================================
# Task 12.1 — Lightly Normalize Tweet Text
# =========================================================

# 注意：
# 这里的 preprocessing 比 TF-IDF 的 preprocessing 更轻。
#
# TF-IDF:
#   会 lowercase、tokenize、remove stop words、
#   lemmatize、remove punctuation 等。
#
# RoBERTa:
#   希望尽可能保留原句信息，
#   所以只统一 username 和 URL。


def prepare_for_roberta(text):

    # 确保输入一定是字符串
    text = str(text)

    # -----------------------------------------------------
    # 1. Normalize usernames
    # -----------------------------------------------------
    #
    # @\w+
    #
    # @     → 匹配 @
    # \w+   → 一个或多个字母 / 数字 / 下划线
    #
    # 例如：
    #
    # @alice
    # @bob123
    #
    # →
    #
    # @user
    #
    # 这样可以保留“这里有一个 mention”的信息，
    # 但不会让具体用户名影响模型。

    text = re.sub(
        r"@\w+",
        "@user",
        text
    )

    # -----------------------------------------------------
    # 2. Normalize URLs
    # -----------------------------------------------------
    #
    # https?://\S+
    #   → http:// 或 https:// 开头的 URL
    #
    # |
    #   → OR
    #
    # www\.\S+
    #   → www. 开头的 URL
    #
    # 所有 URL 最后统一变成：
    #
    # http

    text = re.sub(
        r"https?://\S+|www\.\S+",
        "http",
        text
    )

    # 删除开头和结尾多余空格
    return text.strip()


# =========================================================
# Create sentiment_text column
# =========================================================

df["sentiment_text"] = (

    # 从原始 tweet 开始，
    # 而不是使用 TF-IDF 的 text_clean。
    df["tweet_text_raw"]

    # 如果有 missing text，
    # 先临时替换成空字符串。
    #
    # NaN
    # →
    # ""
    .fillna("")

    # 对每一条 tweet 执行 prepare_for_roberta()
    .apply(prepare_for_roberta)
)


# 可以先看一下处理结果
print(
    df[
        [
            "tweet_text_raw",
            "sentiment_text"
        ]
    ].head()
)


# =========================================================
# Task 12.2 — Analyze All Tweets
# =========================================================

# 将 pandas Series 转成 Python list，
# 一次交给模型处理。
#
# 例如：
#
# [
#   "tweet 1",
#   "tweet 2",
#   "tweet 3"
# ]

results = sentiment_model(

    df["sentiment_text"].tolist(),

    # 如果文本超过模型允许的最大长度，
    # 自动截断。
    truncation=True,

    # 每次让模型同时处理 16 条 tweets，
    # 比一条一条处理通常更高效。
    batch_size=16
)


# =========================================================
# Task 12.3 — Convert Model Output into Dictionaries
# =========================================================

# 模型对于一条 tweet 的输出大概是：
#
# [
#   {"label": "negative", "score": 0.05},
#   {"label": "neutral",  "score": 0.15},
#   {"label": "positive", "score": 0.80}
# ]
#
# 这种 list-of-dictionaries 形式不太方便直接访问。
#
# 我们把它转换成：
#
# {
#   "negative": 0.05,
#   "neutral":  0.15,
#   "positive": 0.80
# }


def scores_to_dict(scores):

    return {
        item["label"].lower(): item["score"]
        for item in scores
    }


# 对每一条 tweet 的 sentiment output
# 都做一次转换。

score_dicts = [
    scores_to_dict(scores)
    for scores in results
]


# =========================================================
# Extract Negative / Neutral / Positive Probabilities
# =========================================================

# 把每个 sentiment probability
# 单独保存成 DataFrame column。


df["sentiment_negative"] = [
    scores.get("negative", 0)
    for scores in score_dicts
]


df["sentiment_neutral"] = [
    scores.get("neutral", 0)
    for scores in score_dicts
]


df["sentiment_positive"] = [
    scores.get("positive", 0)
    for scores in score_dicts
]


# =========================================================
# Choose Final Sentiment Label
# =========================================================

# 对三个 sentiment score：
#
# negative
# neutral
# positive
#
# 找到最大的那个，
# 作为最终 sentiment label。


def predicted_label(scores):

    return max(
        scores,
        key=scores.get
    ).capitalize()


# 例如：
#
# {
#   "negative": 0.05,
#   "neutral": 0.15,
#   "positive": 0.80
# }
#
# 最大的是 positive
#
# →
#
# "Positive"

df["sentiment"] = [
    predicted_label(scores)
    for scores in score_dicts
]


# =========================================================
# Inspect Sentiment Results
# =========================================================

print(
    df[
        [
            "tweet_text_raw",
            "sentiment_negative",
            "sentiment_neutral",
            "sentiment_positive",
            "sentiment"
        ]
    ].head()
)


# =========================================================
# Task 12.4 — Create Numeric Sentiment Score
# =========================================================

# 除了 categorical sentiment：
#
# Positive / Neutral / Negative
#
# 我们还创建一个连续数值：
#
# sentiment_score
# =
# P(positive) - P(negative)
#
#
# 大致范围：
#
# -1 ---------------- 0 ---------------- +1
# Negative          Neutral            Positive


df["sentiment_score"] = (

    df["sentiment_positive"]
    -
    df["sentiment_negative"]

)


# =========================================================
# Inspect Final Result
# =========================================================

print(
    df[
        [
            "tweet_text_raw",
            "sentiment",
            "sentiment_score"
        ]
    ].head()
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test result:
[[{'label': 'positive', 'score': 0.9880043268203735}, {'label': 'neutral', 'score': 0.007510371506214142}, {'label': 'negative', 'score': 0.004485306330025196}]]
                                      tweet_text_raw  \
0  Loved the new dashboard! It is fast, clear, an...   
1  The update is terrible... app crashed twice. h...   
2  Trying the new feature today. Not sure what I ...   
3  Great work @team!!! Version 2.0 feels much smo...   
4  Why is login still broken??? This is so frustr...   

                                      sentiment_text  
0  Loved the new dashboard! It is fast, clear, an...  
1  The update is terrible... app crashed twice. http  
2  Trying the new feature today. Not sure what I ...  
3  Great work @user!!! Version 2.0 feels much smo...  
4  Why is login still broken??? This is so frustr...  
                                      tweet_text_raw  sentiment_negative  \
0  Loved the new dashboard! It is fast, clear, an...            0.003645   
1  The